# Red Neuronal — Screen Time vs Mental Health
Notebook para **Google Colab**. Descarga el dataset directamente desde Kaggle (online) y entrena una red neuronal con Keras/TensorFlow.

Dataset: [Screen Time vs Mental Health (ML-Ready)](https://www.kaggle.com/datasets/kylefengkfeng209/screen-time-vs-mental-health-ml-ready) — 4,810 adolescentes, variables de tiempo de pantalla, sueño y puntaje BDI-II de depresión, con un target de ML ya definido.

> ⚠️ Necesitas una cuenta de Kaggle y tu token de API (`kaggle.json`). Si no lo tienes: Kaggle → Settings → API → *Create New Token*. El Bloque 2 te pedirá subirlo o autenticarte.


## Bloque 1 — Instalación de librerías

In [ ]:
!pip install -q kagglehub tensorflow scikit-learn seaborn pandas matplotlib


## Bloque 2 — Descarga del dataset desde Kaggle (online)
Usamos `kagglehub`, que descarga el dataset directamente desde Kaggle sin necesidad de subir el CSV manualmente.
Al ejecutar esta celda, si no tienes credenciales configuradas, te pedirá autenticarte (usuario + API key).

In [ ]:
import kagglehub

# Descarga el dataset (se cachea localmente tras la primera ejecución)
dataset_path = kagglehub.dataset_download("kylefengkfeng209/screen-time-vs-mental-health-ml-ready")
print("Dataset descargado en:", dataset_path)

import os
archivos = os.listdir(dataset_path)
print("Archivos encontrados:", archivos)


## Bloque 3 — Carga y exploración de datos

In [ ]:
import pandas as pd

# Toma el primer CSV encontrado en la carpeta descargada
csv_file = [f for f in archivos if f.endswith(".csv")][0]
csv_path = os.path.join(dataset_path, csv_file)

df = pd.read_csv(csv_path)
print("Forma del dataset:", df.shape)
df.head()


In [ ]:
df.info()
print("\nValores nulos por columna:\n", df.isnull().sum())
df.describe(include="all")


## Bloque 4 — Definir la variable objetivo (target)
Este dataset viene con un "target de ML ya definido", pero el nombre exacto de la columna puede variar según la versión descargada.
Ejecuta la celda de abajo, **revisa la lista de columnas impresa** y ajusta `TARGET_COLUMN` si es necesario (por defecto se intenta detectar automáticamente).

In [ ]:
print("Columnas disponibles:")
for c in df.columns:
    print(" -", c)

# Intento de detección automática del target (ajusta manualmente si no acierta)
posibles_targets = [c for c in df.columns if any(
    k in c.lower() for k in ["target", "label", "depress", "risk", "bdi", "class"]
)]
print("\nCandidatos detectados como target:", posibles_targets)

# 👉 AJUSTA ESTA LÍNEA con el nombre exacto de tu columna objetivo:
TARGET_COLUMN = posibles_targets[0] if posibles_targets else df.columns[-1]
print("\nUsando como target:", TARGET_COLUMN)
df[TARGET_COLUMN].value_counts()


## Bloque 5 — Preprocesamiento
- Separar features (X) y target (y)
- Codificar variables categóricas
- Escalar variables numéricas
- Split train/test

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df_proc = df.dropna().copy()

X = df_proc.drop(columns=[TARGET_COLUMN])
y = df_proc[TARGET_COLUMN]

# Codificar columnas categóricas de X
cat_cols = X.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Codificar el target si es categórico (clasificación)
es_clasificacion = y.dtype == "object" or y.nunique() <= 10
if y.dtype == "object":
    le_target = LabelEncoder()
    y = le_target.fit_transform(y)
    print("Clases del target:", list(le_target.classes_))

X = X.values.astype("float32")
y = np.array(y).astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if es_clasificacion else None
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Tipo de problema:", "Clasificación" if es_clasificacion else "Regresión")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)


## Bloque 6 — Arquitectura de la red neuronal
Red densa (fully-connected) con Keras. La capa de salida y la función de pérdida se ajustan automáticamente según si el problema es de clasificación binaria, multiclase o regresión.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

n_features = X_train.shape[1]
n_clases = len(np.unique(y_train)) if es_clasificacion else None

model = models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
])

if es_clasificacion and n_clases == 2:
    model.add(layers.Dense(1, activation="sigmoid"))
    loss = "binary_crossentropy"
    metrics = ["accuracy"]
elif es_clasificacion and n_clases > 2:
    model.add(layers.Dense(n_clases, activation="softmax"))
    loss = "sparse_categorical_crossentropy"
    metrics = ["accuracy"]
else:
    model.add(layers.Dense(1, activation="linear"))
    loss = "mse"
    metrics = ["mae"]

model.summary()


## Bloque 7 — Compilación del modelo

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=metrics,
)


## Bloque 8 — Entrenamiento
Con `EarlyStopping` para evitar sobreajuste.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)


## Bloque 9 — Evaluación y visualización de resultados

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Pérdida (loss)")
axes[0].set_xlabel("Época")
axes[0].legend()

metric_key = metrics[0]
axes[1].plot(history.history[metric_key], label="train")
axes[1].plot(history.history[f"val_{metric_key}"], label="val")
axes[1].set_title(metric_key)
axes[1].set_xlabel("Época")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
resultados = model.evaluate(X_test, y_test, verbose=0)
for nombre, valor in zip(model.metrics_names, resultados):
    print(f"{nombre}: {valor:.4f}")

if es_clasificacion:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns

    if n_clases == 2:
        y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
    else:
        y_pred = np.argmax(model.predict(X_test), axis=1)

    print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title("Matriz de confusión")
    plt.show()


## Bloque 10 — Predicciones con nuevos datos y guardado del modelo

In [ ]:
# Ejemplo: predecir sobre las primeras 5 filas del set de prueba
predicciones = model.predict(X_test[:5])
print("Predicciones (crudas):\n", predicciones)

# Guardar el modelo entrenado
model.save("modelo_screen_time_mental_health.keras")
print("\nModelo guardado como modelo_screen_time_mental_health.keras")
